# Définition des constantes

In [2]:
%pip install -r ../requirements.txt

     ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
     -------------------------------------  15.7/15.8 MB 110.3 MB/s eta 0:00:01
     ---------------------------------------- 15.8/15.8 MB 83.5 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [21 lines of output]
      + f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Scripts\python.exe C:\Users\adalc\AppData\Local\Temp\pip-install-da8_ctr_\numpy_09747ecc0c0e41b9991c729d554814d8\vendored-meson\meson\meson.py setup C:\Users\adalc\AppData\Local\Temp\pip-install-da8_ctr_\numpy_09747ecc0c0e41b9991c729d554814d8 C:\Users\adalc\AppData\Local\Temp\pip-install-da8_ctr_\numpy_09747ecc0c0e41b9991c729d554814d8\.mesonpy-7l3fkqxv -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --native-file=C:\Users\adalc\AppData\Local\Temp\pip-install-da8_ctr_\numpy_09747ecc0c0e41b9991c729d554814d8\.mesonpy-7l3fkqxv\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.99
      Source dir: C:\Users\adalc\AppData\Local\Temp\pip-install-da8_ctr_\numpy_09747ecc0c0e41b9991c729d554814d8
      Build dir: C:\Users\adalc\AppData\Local\T

In [7]:
import sys, platform
from pathlib import Path
try:
    import torch
except ModuleNotFoundError:
    %pip install torch
    import torch
try:
    import pandas as pd
except ModuleNotFoundError:
    %pip install pandas
    import pandas as pd

ROOT = Path.cwd().parent

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

DATA = ROOT / "Data"
SRC = ROOT / "SRC"
SPLIT = DATA / "SPLIT"
SPLIT.mkdir(parents=True, exist_ok=True)


# Importation du jeu de données

In [4]:
columns_name = ['TARGET', 'id', 'date', '??', 'user', 'tweet']

df = pd.read_csv(DATA / "training.1600000.processed.noemoticon.csv", encoding='ISO-8859-1', names=columns_name)
print(df.head())

   TARGET          id                          date        ??  \
0       0  1467810369  Mon Apr 06 22:19:45 PDT 2009  NO_QUERY   
1       0  1467810672  Mon Apr 06 22:19:49 PDT 2009  NO_QUERY   
2       0  1467810917  Mon Apr 06 22:19:53 PDT 2009  NO_QUERY   
3       0  1467811184  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   
4       0  1467811193  Mon Apr 06 22:19:57 PDT 2009  NO_QUERY   

              user                                              tweet  
0  _TheSpecialOne_  @switchfoot http://twitpic.com/2y1zl - Awww, t...  
1    scotthamilton  is upset that he can't update his Facebook by ...  
2         mattycus  @Kenichan I dived many times for the ball. Man...  
3          ElleCTF    my whole body feels itchy and like its on fire   
4           Karoli  @nationwideclass no, it's not behaving at all....  


# Train Test Split

In [5]:
from sklearn.model_selection import train_test_split

df["TARGET_BINARY"] = df["TARGET"].map({0: 0, 4: 1})
dftrainval, dftest = train_test_split(df[["TARGET_BINARY", "tweet"]], test_size=50_000, random_state=42, stratify=df["TARGET_BINARY"])

dftrain, dfval = train_test_split(dftrainval, test_size=25_000, random_state=42, stratify=dftrainval["TARGET_BINARY"])


## Sauvegarde des splits

In [8]:

#TRAIN
dftrain.to_csv(SPLIT / "train.csv", index=False, encoding="utf-8")

#VALIDATION
dfval.to_csv(SPLIT / "val.csv", index=False, encoding="utf-8")

#TEST
dftest.to_csv(SPLIT / "test.csv", index=False, encoding="utf-8")

# PIPELINE 1 (LSTM)

## Pré-traitement

### Nettoyage

In [9]:
from SRC.preprocessing import clean_tweet_LSTM

df_train_LSTM = pd.read_csv(SPLIT / "train.csv", encoding='utf-8')
df_train_LSTM = df_train_LSTM.sample(
    n=300_000, random_state=42
).reset_index(drop=True)


df_val_LSTM = pd.read_csv(SPLIT / "val.csv", encoding='utf-8')
df_test_LSTM = pd.read_csv(SPLIT / "test.csv", encoding='utf-8')

df_train_LSTM["tweet_net"] = df_train_LSTM["tweet"].apply(clean_tweet_LSTM)
df_val_LSTM["tweet_net"] = df_val_LSTM["tweet"].apply(clean_tweet_LSTM)
df_test_LSTM["tweet_net"] = df_test_LSTM["tweet"].apply(clean_tweet_LSTM)

print(df_train_LSTM[["tweet", "tweet_net"]].head())

                                               tweet  \
0  Watching Ramsay's Kitchen Nightmares now  Ew c...   
1  On my way too the beachh with thee bitchess  a...   
2  My hand is swollen, bruised and all  the shit ...   
3  @MrBillyBones although it would be the highlig...   
4             Going to the beach with Cody and Jake    

                                           tweet_net  
0  watching ramsays kitchen nightmares now ew coc...  
1  on my way too the beachh with thee bitchess ah...  
2  my hand is swollen bruised and all the shit hu...  
3  although it would be the highlight of the summ...  
4              going to the beach with cody and jake  


### Embedding (avec Glove)

In [10]:
import numpy as np
from collections import Counter

MAX_VOCAB = 30_000

# --- Vocabulaire à partir du train nettoyé ---
counter = Counter()
for txt in df_train_LSTM["tweet_net"]:
    counter.update(str(txt).split())

# index 0 = <pad>, index 1 = <unk>
itos = ["<pad>", "<unk>"] + [w for w, _ in counter.most_common(MAX_VOCAB - 2)]
stoi = {w: i for i, w in enumerate(itos)}
print(f"Vocabulaire : {len(itos):,} tokens")

if not(Path(DATA / "Embedding" / "emb_matrix_300k.npy").exists()):

    EMB_DIM = 200
    GLOVE_PATH = DATA / "Embedding" / "glove.twitter.27B.200d.txt"


    # --- Matrice d'embeddings ---
    rng = np.random.default_rng(42)
    emb_matrix = rng.normal(0, 0.1, (len(itos), EMB_DIM)).astype(np.float32)
    emb_matrix[0] = 0.0  # <pad> à zéro

    found = 0
    with open(GLOVE_PATH, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            idx = stoi.get(word)
            if idx is not None:
                emb_matrix[idx] = np.asarray(parts[1:], dtype=np.float32)
                found += 1

    print(f"Couverture GloVe : {found:,}/{len(itos):,} ({found/len(itos):.1%})")

    np.save(DATA / "Embedding" / "emb_matrix_300k.npy", emb_matrix)
    lengths = df_train_LSTM["tweet_net"].str.split().str.len()
    print(lengths.describe())
    print(f"p95 : {lengths.quantile(0.95):.0f} | p99 : {lengths.quantile(0.99):.0f}")
else:
    emb_matrix = np.load(DATA / "Embedding" / "emb_matrix_300k.npy")

Vocabulaire : 30,000 tokens


## Entrainement

### tweets => Matrices

In [11]:
import numpy as np
import torch

MAX_LEN = 40
PAD, UNK = 0, 1

def textes_vers_matrice(series_textes):
    """Convertit une colonne de textes en matrice (n_tweets, 40)."""
    matrice = np.zeros((len(series_textes), MAX_LEN), dtype=np.int64)
    for i, texte in enumerate(series_textes):
        mots = str(texte).split()[:MAX_LEN]
        for j, mot in enumerate(mots):
            matrice[i, j] = stoi.get(mot, UNK)
    return matrice

# Conversion en tenseurs
X_train = torch.tensor(textes_vers_matrice(df_train_LSTM["tweet_net"]))
y_train = torch.tensor(df_train_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

X_val = torch.tensor(textes_vers_matrice(df_val_LSTM["tweet_net"]))
y_val = torch.tensor(df_val_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

X_test = torch.tensor(textes_vers_matrice(df_test_LSTM["tweet_net"]))
y_test = torch.tensor(df_test_LSTM["TARGET_BINARY"].values, dtype=torch.float32)

print(X_train.shape, y_train.shape)

torch.Size([300000, 40]) torch.Size([300000])


### Modèle

In [12]:
import torch.nn as nn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

class LSTMSentiment(nn.Module):
    def __init__(self, emb_matrix):
        super().__init__()
        # 1. Embedding : indice -> vecteur GloVe de dim 200
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(emb_matrix), freeze=False, padding_idx=PAD
        )
        # 2. LSTM bidirectionnel : lit le tweet dans les deux sens
        self.lstm = nn.LSTM(200, 128, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        # 3. Sortie : 256 -> 1 score
        self.fc = nn.Linear(256, 1)

    def forward(self, x):
        emb = self.embedding(x)              # (batch, 40, 200)
        sorties, _ = self.lstm(emb)          # (batch, 40, 256)
        moyenne = sorties.mean(dim=1)        # (batch, 256)
        return self.fc(self.dropout(moyenne)).squeeze(1)

model = LSTMSentiment(emb_matrix).to(DEVICE)

cpu


### boucle d'entrainement

In [13]:
from sklearn.metrics import accuracy_score, f1_score
import time, copy

best_f1, best_state, patience = 0, None, 0
BATCH = 256
EPOCHS = 5

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def predire(X):
    model.eval()
    predictions = []
    with torch.no_grad():
        for i in range(0, len(X), 512):
            logits = model(X[i:i+512].to(DEVICE))
            predictions += (torch.sigmoid(logits) > 0.5).long().cpu().tolist()
    return predictions


for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(X_train))     # mélange à chaque epoch
    perte_totale, t0 = 0, time.time()

    for i in range(0, len(X_train), BATCH):
        idx = perm[i:i+BATCH]
        xb, yb = X_train[idx].to(DEVICE), y_train[idx].to(DEVICE)

        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        perte_totale += loss.item()
    preds = predire(X_val)
    f1 = f1_score(y_val, preds, average="macro")
    acc = accuracy_score(y_val, preds)
    print(f"Epoch {epoch} | loss {perte_totale/(len(X_train)//BATCH):.4f} | "
          f"val acc {acc:.4f} | val F1 {f1:.4f} | {time.time()-t0:.0f}s")

    if f1 > best_f1:
        best_f1, best_state, patience = f1, copy.deepcopy(model.state_dict()), 0
    else:
        patience += 1
        if patience >= 2:
            print(f"Early stopping — meilleur F1 val : {best_f1:.4f}")
            break

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Modèle restauré — F1 val : {best_f1:.4f}")

Epoch 1 | loss 0.4491 | val acc 0.8090 | val F1 0.8090 | 148s
Epoch 2 | loss 0.3897 | val acc 0.8192 | val F1 0.8191 | 161s
Epoch 3 | loss 0.3542 | val acc 0.8190 | val F1 0.8189 | 160s
Epoch 4 | loss 0.3180 | val acc 0.8177 | val F1 0.8177 | 178s
Early stopping — meilleur F1 val : 0.8191
Modèle restauré — F1 val : 0.8191


## Tests

In [14]:
from sklearn.metrics import confusion_matrix, classification_report

preds_test = predire(X_test)
print(confusion_matrix(y_test, preds_test))
print(classification_report(y_test, preds_test, target_names=["négatif", "positif"]))

(ROOT / "Models").mkdir(exist_ok=True)

torch.save({
    "state_dict": model.state_dict(),
    "itos": itos,
    "stoi": stoi,
    "max_len": MAX_LEN,
}, ROOT / "Models" / "lstm_baseline.pt")

[[20865  4135]
 [ 5032 19968]]
              precision    recall  f1-score   support

     négatif       0.81      0.83      0.82     25000
     positif       0.83      0.80      0.81     25000

    accuracy                           0.82     50000
   macro avg       0.82      0.82      0.82     50000
weighted avg       0.82      0.82      0.82     50000



# PIPELINE 2 (SETFIT)

## Pré-traitement

### Nettoyage

In [ ]:
from SRC.preprocessing import clean_tweet_SETFIT
from datasets import Dataset

df_train_SETFIT = pd.read_csv(SPLIT / "train.csv", encoding='utf-8')


# Rechargement des splits (nettoyage SetFit déjà appliqué au train)
df_val_SETFIT = pd.read_csv(SPLIT / "val.csv", encoding="utf-8")
df_test_SETFIT = pd.read_csv(SPLIT / "test.csv", encoding="utf-8")

df_val_SETFIT["tweet_net"] = df_val_SETFIT["tweet"].apply(clean_tweet_SETFIT)
df_test_SETFIT["tweet_net"] = df_test_SETFIT["tweet"].apply(clean_tweet_SETFIT)

# --- Échantillonnage few-shot : N exemples par classe ---
N_PER_CLASS = 64

df_fewshot = (
    df_train_SETFIT
    .groupby("TARGET_BINARY")
    .sample(n=N_PER_CLASS, random_state=42)
    .reset_index(drop=True)
)
df_fewshot["tweet_net"] = df_fewshot["tweet"].apply(clean_tweet_SETFIT)

# Sous-échantillon de validation (SetFit n'a pas besoin de 25k pour valider)
df_val_small = df_val_SETFIT.sample(n=2_000, random_state=42).reset_index(drop=True)

print(f"Train few-shot : {len(df_fewshot)} exemples "
      f"({N_PER_CLASS} par classe)")
print(f"Validation     : {len(df_val_small)}")
print(f"Test           : {len(df_test_SETFIT)}")

# Conversion au format HuggingFace Dataset
train_ds = Dataset.from_pandas(
    df_fewshot[["tweet_net", "TARGET_BINARY"]]
    .rename(columns={"tweet_net": "text", "TARGET_BINARY": "label"})
)
val_ds = Dataset.from_pandas(
    df_val_small[["tweet_net", "TARGET_BINARY"]]
    .rename(columns={"tweet_net": "text", "TARGET_BINARY": "label"})
)



Train few-shot : 64 exemples (32 par classe)
Validation     : 2000
Test           : 50000


## Entrainement

In [19]:
from setfit import SetFitModel, Trainer, TrainingArguments
import time

MODEL_NAME = "sentence-transformers/paraphrase-mpnet-base-v2"

model_setfit = SetFitModel.from_pretrained(
    MODEL_NAME,
    labels=["négatif", "positif"],
)

args = TrainingArguments(
    batch_size=16,
    num_epochs=1,              # epochs du corps (contrastif)
    num_iterations=20,         # paires générées par exemple
    seed=42,
    report_to="none",
)

trainer = Trainer(
    model=model_setfit,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    metric="accuracy",
)

t0 = time.time()
trainer.train()
print(f"Entraînement terminé en {time.time() - t0:.0f}s")

metrics = trainer.evaluate()
print(metrics)

f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\adalc\.cache\huggingface\hub\models--sentence-transformers--paraphrase-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' p

Step,Training Loss
1,0.441900
50,0.225900
100,0.009700
150,0.001000


***** Running evaluation *****


Entraînement terminé en 418s


{'accuracy': 0.7505}


## Tests

In [20]:
from sklearn.metrics import (
    accuracy_score, f1_score,
    confusion_matrix, classification_report
)
import numpy as np

def predire_setfit(textes, batch=256):
    """Prédiction par lots pour éviter la saturation mémoire."""
    preds = []
    for i in range(0, len(textes), batch):
        sortie = model_setfit.predict(list(textes[i:i+batch]))
        preds.extend(sortie)
    return np.array(preds)

textes_test = df_test_SETFIT["tweet_net"].tolist()
y_test_setfit = df_test_SETFIT["TARGET_BINARY"].values

t0 = time.time()
preds_setfit = predire_setfit(textes_test)
print(f"Inférence sur {len(textes_test):,} tweets : {time.time() - t0:.0f}s")

# Les labels reviennent en chaînes -> remise en binaire
if preds_setfit.dtype.kind in "OU":
    preds_setfit = np.where(preds_setfit == "positif", 1, 0)

print(confusion_matrix(y_test_setfit, preds_setfit))
print(classification_report(
    y_test_setfit, preds_setfit,
    target_names=["négatif", "positif"]
))

Inférence sur 50,000 tweets : 366s
[[20941  4059]
 [ 7517 17483]]
              precision    recall  f1-score   support

     négatif       0.74      0.84      0.78     25000
     positif       0.81      0.70      0.75     25000

    accuracy                           0.77     50000
   macro avg       0.77      0.77      0.77     50000
weighted avg       0.77      0.77      0.77     50000



## Sauvegarde

In [21]:
SETFIT_DIR = ROOT / "Models" / "setfit_poc"
SETFIT_DIR.mkdir(parents=True, exist_ok=True)

model_setfit.save_pretrained(str(SETFIT_DIR))
print(f"Modèle sauvegardé dans {SETFIT_DIR}")

Modèle sauvegardé dans f:\adalc\Documents\Projets\Projet 9\OpenClassrooms_projet_9\Models\setfit_poc


In [22]:
import pandas as pd

resultats = pd.DataFrame([
    {
        "Modèle": "LSTM + GloVe (baseline)",
        "Exemples d'entraînement": 300_000,
        "Accuracy": accuracy_score(y_test, preds_test),
        "F1 macro": f1_score(y_test, preds_test, average="macro"),
    },
    {
        "Modèle": f"SetFit ({N_PER_CLASS}/classe)",
        "Exemples d'entraînement": len(df_fewshot),
        "Accuracy": accuracy_score(y_test_setfit, preds_setfit),
        "F1 macro": f1_score(y_test_setfit, preds_setfit, average="macro"),
    },
])

resultats["Accuracy"] = resultats["Accuracy"].round(4)
resultats["F1 macro"] = resultats["F1 macro"].round(4)

print(resultats.to_string(index=False))
resultats.to_csv(ROOT / "Models" / "comparaison.csv", index=False)

                 Modèle  Exemples d'entraînement  Accuracy  F1 macro
LSTM + GloVe (baseline)                   300000    0.8167    0.8166
     SetFit (32/classe)                       64    0.7685    0.7674
